In [1]:
import calendar
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.max_row',None)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 1, 10), datetime.date(2023, 1, 9))

In [2]:
# specify the number of business days
num_days = 1
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(num_days)
yesterday = today - num_business_days
#yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-10
yesterday: 2023-01-09 00:00:00


In [3]:
yesterday = yesterday.date()
a_year_ago = yesterday - timedelta(days=365)
yesterday, a_year_ago

(datetime.date(2023, 1, 9), datetime.date(2022, 1, 9))

### Get past one quarter data

In [4]:
sql = """
SELECT name
FROM buy 
ORDER BY name
"""
df = pd.read_sql(sql, const)

names = df['name'].values.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))
in_p

"'ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART'"

In [5]:
in_p = "'SINGER'"
in_p

"'SINGER'"

In [6]:
one_qtr_ago = yesterday - timedelta(days=20)
one_qtr_ago

sql = """
SELECT * 
FROM price 
WHERE date >= '%s' AND name IN (%s) 
ORDER BY name, date"""
sql = sql % (one_qtr_ago, in_p)
print(sql)


SELECT * 
FROM price 
WHERE date >= '2022-12-20' AND name IN ('SINGER') 
ORDER BY name, date


In [7]:
df = pd.read_sql(sql, const)
df.shape

(14, 7)

In [8]:
data_path = "../data/"
file_name = "Quarterly-Price-by-Name.csv"
output_file = data_path + file_name

df.set_index("name", inplace=True)
df.to_csv(output_file, header=None)

### Command Line Call Lwr-Dly & Upr-Dly Process

In [9]:
name = 'SINGER'

In [10]:
data_path = "../csv/"
file_name   = name + '-lower.csv'
input_file = data_path + file_name
col_names   = ['name','fm_date','to_date','days','fm_price','to_price','chg_amt','chg_pct','spd','avg','maxp','minp','qty']
df_lower = pd.read_csv(input_file, sep=',',  index_col=None, names=col_names)
df_lower.sort_values(by=['to_date'],ascending=[False])

,name,fm_date,to_date,days,fm_price,to_price,chg_amt,chg_pct,spd,avg,maxp,minp,qty
1,SINGER,2022-12-29,2023-01-06,6,28.75,27.00,-1.75,-6.09,-7,-2,28.75,26.75,30985321
0,SINGER,2022-12-20,2022-12-23,3,29.25,27.75,-1.50,-5.13,-6,-2,29.50,27.00,31271427


In [11]:
df_lower[df_lower['chg_pct'] <= -5].sort_values(by=['to_date'],ascending=[False])

,name,fm_date,to_date,days,fm_price,to_price,chg_amt,chg_pct,spd,avg,maxp,minp,qty
1,SINGER,2022-12-29,2023-01-06,6,28.75,27.00,-1.75,-6.09,-7,-2,28.75,26.75,30985321
0,SINGER,2022-12-20,2022-12-23,3,29.25,27.75,-1.50,-5.13,-6,-2,29.50,27.00,31271427


In [12]:
data_path = "../csv/"
file_name   = name + '-upper.csv'
input_file = data_path + file_name
col_names   = ['name','fm_date','to_date','days','fm_price','to_price','chg_amt','chg_pct','spd','avg','maxp','minp','qty']
df_upper = pd.read_csv(input_file, sep=',',  index_col=None, names=col_names)
df_upper.sort_values(by=['to_date'],ascending=[False])

,name,fm_date,to_date,days,fm_price,to_price,chg_amt,chg_pct,spd,avg,maxp,minp,qty
1,SINGER,2023-01-05,2023-01-09,2,27.00,29.75,2.75,10.19,11,5,30.00,27.0,21930361
0,SINGER,2022-12-23,2022-12-30,5,27.75,28.75,1.00,3.60,4,0,29.25,27.5,27255168


In [13]:
df_upper[df_upper['chg_pct'] >= 5].sort_values(by=['to_date'],ascending=[False])

,name,fm_date,to_date,days,fm_price,to_price,chg_amt,chg_pct,spd,avg,maxp,minp,qty
1,SINGER,2023-01-05,2023-01-09,2,27.0,29.75,2.75,10.19,11,5,30.0,27.0,21930361


### Concat

In [14]:
frames = [df_lower, df_upper]
result = pd.concat(frames)
result.sort_values(by=['to_date'],ascending=[False])

,name,fm_date,to_date,days,fm_price,to_price,chg_amt,chg_pct,spd,avg,maxp,minp,qty
1,SINGER,2023-01-05,2023-01-09,2,27.00,29.75,2.75,10.19,11,5,30.00,27.00,21930361
1,SINGER,2022-12-29,2023-01-06,6,28.75,27.00,-1.75,-6.09,-7,-2,28.75,26.75,30985321
0,SINGER,2022-12-23,2022-12-30,5,27.75,28.75,1.00,3.60,4,0,29.25,27.50,27255168
0,SINGER,2022-12-20,2022-12-23,3,29.25,27.75,-1.50,-5.13,-6,-2,29.50,27.00,31271427


### Sales table in port_lite

In [ ]:
sql = """
SELECT *
FROM sales
WHERE to_date = '%s'"""
sql = sql % today
print(sql)
df_sales = pd.read_sql(sql, conlite)
df_sales

In [ ]:
df_sales[df_sales['pct'] >= 5].sort_values(by=['pct'],ascending=[False])

In [ ]:
df_sales[df_sales['pct'] <= -5].sort_values(by=['pct'],ascending=[True])